<a href="https://colab.research.google.com/github/yash99-bot/Drive__car/blob/main/Customer%20Engagement%20%26%20Product%20Utilization%20Analytics%20for%20Retention%20StrategyUntitled3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
data = pd.read_csv(r'European_Bank.csv')

In [ ]:
data.head()

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Unnamed: 14,Unnamed: 15
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,NaN,NaN
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,NaN,NaN
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,NaN,NaN
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,NaN,NaN
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,NaN,NaN


In [ ]:
data.index

RangeIndex(start=0, stop=10000, step=1)

In [ ]:
data.columns

Index(['Year', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender',
       'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited', 'Unnamed: 14',
       'Unnamed: 15'],
      dtype='object')

In [ ]:
data.isnull().sum()

,0
Year,0
CustomerId,0
Surname,0
CreditScore,0
Geography,0
Gender,0
Age,0
Tenure,0
Balance,0
NumOfProducts,0


In [ ]:
data.shape

(10000, 16)

In [ ]:
data.duplicated().sum()

np.int64(0)

In [ ]:
data.columns.tolist()

['Year',
 'CustomerId',
 'Surname',
 'CreditScore',
 'Geography',
 'Gender',
 'Age',
 'Tenure',
 'Balance',
 'NumOfProducts',
 'HasCrCard',
 'IsActiveMember',
 'EstimatedSalary',
 'Exited',
 'Unnamed: 14',
 'Unnamed: 15']

In [ ]:
data['NumOfProducts'].value_counts(dropna=False)

,count
NumOfProducts,
1,5084
2,4590
3,266
4,60


In [ ]:
data['IsActiveMember'].value_counts(dropna=False)

,count
IsActiveMember,
1,5151
0,4849


In [ ]:
invalid_products = data[data['NumOfProducts'] <= 0]
print(f"\nRows with invalid product counts (<= 0): {len(invalid_products)}")


Rows with invalid product counts (<= 0): 0


In [ ]:
binary_cols = ['HasCrCard', 'IsActiveMember', 'Exited']
expected_values = {0, 1}
for col in binary_cols:
    unique_vals = set(data[col].dropna().unique())
    is_consistent = unique_vals.issubset(expected_values)
    missing_or_null = data[col].isnull().sum()

    print(f"Variable: '{col}'")
    print(f"  - Unique values found: {unique_vals}")
    print(f"  - Consistent (Only 0 or 1): {is_consistent}")
    print(f"  - Missing/Null entries: {missing_or_null}")

Variable: 'HasCrCard'
  - Unique values found: {np.int64(0), np.int64(1)}
  - Consistent (Only 0 or 1): True
  - Missing/Null entries: 0
Variable: 'IsActiveMember'
  - Unique values found: {np.int64(0), np.int64(1)}
  - Consistent (Only 0 or 1): True
  - Missing/Null entries: 0
Variable: 'Exited'
  - Unique values found: {np.int64(0), np.int64(1)}
  - Consistent (Only 0 or 1): True
  - Missing/Null entries: 0


In [ ]:
# Calculate churn rate by number of products
# Churn rate = mean of 'Exited' if Exited is 1 for churned, 0 for retained
churn_rate_by_prod = data.groupby('NumOfProducts')['Exited'].agg(['count', 'sum', 'mean']).reset_index()
churn_rate_by_prod.rename(columns={'count': 'Total Customers', 'sum': 'Churned Customers', 'mean': 'Churn Rate'}, inplace=True)
churn_rate_by_prod['Churn Rate (%)'] = (churn_rate_by_prod['Churn Rate'] * 100).round(2)
print("\nChurn Rate Breakdown:")
print(churn_rate_by_prod)


Churn Rate Breakdown:
   NumOfProducts  Total Customers  Churned Customers  Churn Rate  \
0              1             5084               1409    0.277144   
1              2             4590                348    0.075817   
2              3              266                220    0.827068   
3              4               60                 60    1.000000   

   Churn Rate (%)  
0           27.71  
1            7.58  
2           82.71  
3          100.00  


In [ ]:
print("\n--- Confirming Churn Labeling Accuracy ---")
# Check if the primary target label 'Exited' has missing data
missing_churn = data['Exited'].isnull().sum()
print(f"Missing values in 'Exited' column: {missing_churn}")
churn_by_activity = data.groupby('IsActiveMember')['Exited'].value_counts(normalize=True).unstack()
print("\nChurn Rate Breakdown by Active Membership status (Proportions):")
print(churn_by_activity)
churn_by_products = data.groupby('NumOfProducts')['Exited'].value_counts(normalize=True).unstack()
print("\nChurn Rate Breakdown by Number of Products (Proportions):")
print(churn_by_products)



--- Confirming Churn Labeling Accuracy ---
Missing values in 'Exited' column: 0

Churn Rate Breakdown by Active Membership status (Proportions):
Exited                 0         1
IsActiveMember                    
0               0.731491  0.268509
1               0.857309  0.142691

Churn Rate Breakdown by Number of Products (Proportions):
Exited                0         1
NumOfProducts                    
1              0.722856  0.277144
2              0.924183  0.075817
3              0.172932  0.827068
4                   NaN  1.000000


In [ ]:
active_engaged_df = data[(data['IsActiveMember'] == 1) & (data['Exited'] == 0)]
print(f"Total Active & Engaged Customers: {len(active_engaged_df)}")

Total Active & Engaged Customers: 4416


In [ ]:
from google.colab import files
# Trigger browser download
files.download('Active_Engaged_Customers.csv')

FileNotFoundError: Cannot find file: Active_Engaged_Customers.csv

In [ ]:
dormant_customers = data[(data['IsActiveMember'] == 0) & (data['Exited'] == 0)]
print(f"Total Dormant/At-Risk Customers: {len(dormant_customers)}")

Total Dormant/At-Risk Customers: 3547


In [ ]:
dormant_customers.to_csv('Inactive_Disengaged_Customers.csv', index=False)
files.download('Inactive_Disengaged_Customers.csv')

In [ ]:
high_bal_inactive_df = data[
    (data['IsActiveMember'] == 0) &
    (data['Exited'] == 0) &
    (data['Balance'] > 100000)
]

print(f"Total Inactive High-Balance Customers: {len(high_bal_inactive_df)}")

Total Inactive High-Balance Customers: 1584


In [ ]:
active_low_product_df = data[
    (data['IsActiveMember'] == 1) &
    (data['Exited'] == 0) &
    (data['NumOfProducts'] == 1)
]

# Display total count and the first few target rows
print(f"Total Active, Low-Product Customers: {len(active_low_product_df)}")

Total Active, Low-Product Customers: 2078


In [ ]:
churn_rate_by_prod = data.groupby('NumOfProducts')['Exited'].agg(['count', 'sum', 'mean']).reset_index()
churn_rate_by_prod.rename(columns={'count': 'Total Customers', 'sum': 'Churned Customers', 'mean': 'Churn Rate'}, inplace=True)
print(churn_rate_by_prod)

   NumOfProducts  Total Customers  Churned Customers  Churn Rate
0              1             5084               1409    0.277144
1              2             4590                348    0.075817
2              3              266                220    0.827068
3              4               60                 60    1.000000


In [ ]:
from google.colab import files
churn_rate_by_prod.to_csv('churn_rate_by_prod.csv', index=False)
files.download('churn_rate_by_prod.csv')

NameError: name 'churn_rate_by_prod' is not defined

In [ ]:
data['Product_Segment'] = data['NumOfProducts'].apply(
    lambda x: 'Single-Product (1)' if x == 1 else 'Multi-Product (2+)'
)
data['Is_Retained'] = 1 - data['Exited']
print(data['Product_Segment'])
print(data['Is_Retained'])

0       Single-Product (1)
1       Single-Product (1)
2       Multi-Product (2+)
3       Multi-Product (2+)
4       Single-Product (1)
               ...        
9995    Multi-Product (2+)
9996    Single-Product (1)
9997    Single-Product (1)
9998    Multi-Product (2+)
9999    Single-Product (1)
Name: Product_Segment, Length: 10000, dtype: object
0       0
1       1
2       0
3       1
4       1
       ..
9995    1
9996    1
9997    0
9998    0
9999    1
Name: Is_Retained, Length: 10000, dtype: int64


In [ ]:
retention_summary = data.groupby('Product_Segment')['Is_Retained'].agg(
    Total_Customers='count',
    Retained_Customers='sum',
    Retention_Rate=lambda x: round(x.mean() * 100, 2)
).reset_index()

print(retention_summary)

      Product_Segment  Total_Customers  Retained_Customers  Retention_Rate
0  Multi-Product (2+)             4916                4288           87.23
1  Single-Product (1)             5084                3675           72.29


In [ ]:
correlation = data['NumOfProducts'].corr(data['Exited'])
print(f"Correlation: {correlation}")
depth_churn = data.groupby('NumOfProducts')['Exited'].agg(['count', 'sum', 'mean']).reset_index()
depth_churn.columns = ['Product_Depth', 'Total_Customers', 'Churned_Customers', 'Churn_Rate']
print(depth_churn)

Correlation: -0.04781986494944055
   Product_Depth  Total_Customers  Churned_Customers  Churn_Rate
0              1             5084               1409    0.277144
1              2             4590                348    0.075817
2              3              266                220    0.827068
3              4               60                 60    1.000000


In [ ]:
balance_by_activity = data.groupby('IsActiveMember')['Balance'].describe()
print("\nBalance stats by Activity Status:")
print(balance_by_activity)
data['Has_Zero_Balance'] = data['Balance'] == 0
zero_bal_cross = pd.crosstab(data['IsActiveMember'], data['Has_Zero_Balance'], normalize='index') * 100
print("\nZero Balance % by Activity Status:")
print(zero_bal_cross)


Balance stats by Activity Status:
                 count          mean           std  min  25%       50%  \
IsActiveMember                                                           
0               4849.0  77134.376863  62545.195018  0.0  0.0  98263.46   
1               5151.0  75875.422145  62257.859389  0.0  0.0  96166.88   

                      75%        max  
IsActiveMember                        
0               128270.56  222267.63  
1               127033.24  250898.09  

Zero Balance % by Activity Status:
Has_Zero_Balance      False      True 
IsActiveMember                        
0                 64.033821  35.966179
1                 63.638129  36.361871


In [ ]:
print("Balance Describe:\n", data['Balance'].describe())
print("CreditScore Describe:\n", data['CreditScore'].describe())
premium_tier = data['Balance'].quantile(0.75)
print(f"Premium Balance Threshold (75th percentile): {premium_tier}")
at_risk_premium = data[(data['Exited'] == 0) & (data['IsActiveMember'] == 0) & (data['Balance'] >= premium_tier)]
print(f"Number of At-Risk Premium Customers: {len(at_risk_premium)}")
print(at_risk_premium[['CustomerId', 'Surname', 'CreditScore', 'Balance', 'Age']].head())

Balance Describe:
 count     10000.000000
mean      76485.889288
std       62397.405202
min           0.000000
25%           0.000000
50%       97198.540000
75%      127644.240000
max      250898.090000
Name: Balance, dtype: float64
CreditScore Describe:
 count    10000.000000
mean       650.528800
std         96.653299
min        350.000000
25%        584.000000
50%        652.000000
75%        718.000000
max        850.000000
Name: CreditScore, dtype: float64
Premium Balance Threshold (75th percentile): 127644.24
Number of At-Risk Premium Customers: 867
    CustomerId    Surname  CreditScore    Balance  Age
51    15768193  Trevisani          585  146050.97   36
56    15630053       Tsao          656  127864.40   45
60    15651280     Hunter          742  136857.00   35
61    15773469      Clark          687  152328.88   27
75    15780961   Cavenagh          735  178718.19   21


In [ ]:
from google.colab import files
sticky_mask = (data['IsActiveMember'] == 1) & (data['NumOfProducts'] == 2)
sticky_customers = data[sticky_mask]
print(f"Total Sticky Customers: {len(sticky_customers)}")
print(f"Churn Rate within this segment: {sticky_customers['Exited'].mean() * 100:.2f}%\n")
sticky_profile = sticky_customers[['Age', 'Tenure', 'CreditScore', 'Balance', 'EstimatedSalary']].mean().to_frame(name='Average_Value')
print(sticky_profile)
sticky_customers.to_csv('Sticky_Customer_Profiles.csv', index=False)
files.download('Sticky_Customer_Profiles.csv')


Total Sticky Customers: 2446
Churn Rate within this segment: 5.56%

                 Average_Value
Age                  39.086672
Tenure                4.957890
CreditScore         655.047016
Balance           51852.938729
EstimatedSalary  100468.101292


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>